# 금융상품 에이전트 단계별 구축 실습

이 노트북은 `주요 참고 정보`의 설계 원칙과 `financial_product_agents`의 구현을 바탕으로, 작은 금융상품 검색 에이전트를 바닥부터 조립합니다.

전체 흐름은 **질의 → 구조화된 계획 → 허용된 CSV 조회 → 근거 검증 → 답변**입니다. LLM이 숫자를 계산하거나 상품 정보를 지어내지 않고, 데이터 엔진이 반환한 값만 설명하게 만드는 것이 핵심입니다.

> 이 실습은 교육용 데이터 조회 예제이며 투자 권유가 아닙니다. 데이터 기준일은 2026-07-11입니다.

## 실습 지도

1. 실행 환경과 데이터 확인
2. 출력 계약(State)을 dataclass로 정의
3. 자연어 질의를 안전한 `QueryPlan`으로 변환
4. 금융상품 온톨로지로 용어·관계·제약 정의
5. 허용 목록 기반 CSV 검색 도구 구현
6. 상품군별 전문 에이전트와 근거 검증 구성
7. 오케스트레이터로 전체 흐름 연결
8. 실패·경계 사례와 기존 구현 비교

각 코드 셀은 앞 셀의 결과를 사용하므로 위에서부터 실행하세요.

In [24]:
from __future__ import annotations

import csv
import json
import re
import sys
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any

# 노트북을 어느 폴더에서 열어도 데이터 경로가 깨지지 않도록 저장소 루트를 탐색합니다.
def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'financial_product_agents').is_dir() and (candidate / '2.금융상품_데이터').is_dir():
            return candidate
    raise FileNotFoundError('저장소 루트를 찾지 못했습니다.')

REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / '2.금융상품_데이터'
print('저장소:', REPO_ROOT)
print('샘플 데이터:', [p.name for p in sorted(DATA_DIR.glob('*_샘플.csv'))])

저장소: /Users/park_jimin/Documents/ai패스티벌
샘플 데이터: ['공모펀드_샘플.csv', '국내ETF_샘플.csv', '국내채권_샘플.csv', '해외ETF_샘플.csv']


## 1단계 — 데이터 계약 정의

에이전트 사이에서 자연어를 주고받으면 해석이 흔들립니다. 필터, 검색 계획, 결과, 최종 응답을 명시적인 자료형으로 고정합니다. `QueryPlan`은 LLM을 나중에 연결하더라도 반드시 검증해야 하는 경계입니다.

In [9]:
# frozen=True로 만든 필터는 생성 후 값이 바뀌지 않아 실행 기록을 신뢰하기 쉽습니다.
@dataclass(frozen=True)
class Filter:
    field: str
    operator: str
    value: str | float | int

@dataclass
class QueryPlan:
    product_type: str | None
    filters: list[Filter] = field(default_factory=list)
    sort_by: str | None = None
    sort_order: str = 'desc'
    limit: int = 5
    clarification: str | None = None
    original_query: str = ''

    # 모델 또는 사용자가 만든 계획이 안전한 범위인지 실행 전에 검증합니다.
    def __post_init__(self):
        if self.sort_order not in {'asc', 'desc'}:
            raise ValueError('sort_order는 asc 또는 desc여야 합니다.')
        self.limit = max(1, min(int(self.limit), 20))
        allowed = {'eq', 'contains', 'gte', 'lte'}
        if any(item.operator not in allowed for item in self.filters):
            raise ValueError('허용되지 않은 필터 연산자가 있습니다.')

@dataclass
class ProductResult:
    product_type: str
    product_id: str
    name: str
    attributes: dict[str, Any]
    source_file: str
    source_row: int

@dataclass
class AgentResponse:
    answer: str
    query_plan: QueryPlan
    results: list[ProductResult] = field(default_factory=list)
    evidence: list[dict[str, Any]] = field(default_factory=list)
    warnings: list[str] = field(default_factory=list)
    data_as_of: str = '2026-07-11'

    def to_dict(self):
        return asdict(self)

QueryPlan('overseas_etf', sort_by='fee', sort_order='asc', limit=3)

QueryPlan(product_type='overseas_etf', filters=[], sort_by='fee', sort_order='asc', limit=3, clarification=None, original_query='')

## 2단계 — 질의 해석 에이전트

첫 버전은 LLM 대신 결정론적 규칙을 씁니다. 작은 규칙 기반 해석기는 저렴하고 재현 가능하며, 이후 OpenAI API의 구조화 출력을 붙일 때 기준선 역할을 합니다. 모호하거나 데이터에 없는 요청은 검색을 강행하지 않고 질문을 되돌려줍니다.

In [25]:
class QueryInterpreterAgent:
    PRODUCT_WORDS = {
        '국내채권': 'domestic_bond', '채권': 'domestic_bond',
        '국내 etf': 'domestic_etf', '국내etf': 'domestic_etf',
        '해외 etf': 'overseas_etf', '해외etf': 'overseas_etf',
        '공모펀드': 'public_fund', '펀드': 'public_fund',
    }
    SORT_WORDS = {
        '총보수': 'fee', '보수': 'fee', 'aum': 'aum', '순자산': 'aum',
        '1개월 수익률': 'return_1m', '1년 수익률': 'return_1y',
        '세후수익률': 'after_tax_yield', '매수수익률': 'buy_yield', '표면금리': 'coupon_rate',
    }

    # 자연어를 바로 실행하지 않고 제한된 구조의 QueryPlan으로 번역합니다.
    def interpret(self, query: str) -> QueryPlan:
        text = ' '.join(query.lower().split())
        product_type = next((v for word, v in self.PRODUCT_WORDS.items() if word in text), None)
        count = re.search(r'(\d+)\s*(?:개|종|건)', text)
        limit = int(count.group(1)) if count else 5
        # 상품군이 모호하면 임의로 추측하지 않고 사용자에게 명확화를 요청합니다.
        if not product_type:
            return QueryPlan(None, limit=limit, clarification='국내채권, 국내 ETF, 해외 ETF, 공모펀드 중 상품군을 지정해 주세요.', original_query=query)
        if product_type == 'public_fund' and '보수' in text:
            return QueryPlan(product_type, limit=limit, clarification='제공된 공모펀드 데이터에는 보수 정보가 없습니다. 다른 기준을 선택해 주세요.', original_query=query)
        sort_by = next((v for word, v in self.SORT_WORDS.items() if word in text), None)
        ascending = any(word in text for word in ('낮은', '저렴', '적은', '짧은'))
        filters = []
        if '미국' in text:
            filters.append(Filter('region', 'contains', 'United States' if product_type == 'overseas_etf' else '미국'))
        return QueryPlan(product_type, filters, sort_by, 'asc' if ascending else 'desc', limit, original_query=query)

interpreter = QueryInterpreterAgent()
for query in ['총보수가 낮은 해외 ETF 3개', '보수가 낮은 공모펀드 3개', '수익률 높은 상품 5개']:
    print(query, '->', interpreter.interpret(query))

총보수가 낮은 해외 ETF 3개 -> QueryPlan(product_type='overseas_etf', filters=[], sort_by='fee', sort_order='asc', limit=3, clarification=None, original_query='총보수가 낮은 해외 ETF 3개')
보수가 낮은 공모펀드 3개 -> QueryPlan(product_type='public_fund', filters=[], sort_by=None, sort_order='desc', limit=3, clarification='제공된 공모펀드 데이터에는 보수 정보가 없습니다. 다른 기준을 선택해 주세요.', original_query='보수가 낮은 공모펀드 3개')
수익률 높은 상품 5개 -> QueryPlan(product_type=None, filters=[], sort_by=None, sort_order='desc', limit=5, clarification='국내채권, 국내 ETF, 해외 ETF, 공모펀드 중 상품군을 지정해 주세요.', original_query='수익률 높은 상품 5개')


## 3단계 — 상품 카탈로그와 허용 목록

사용자가 임의의 컬럼명이나 연산을 실행하지 못하게 상품군별 허용 필드를 정의합니다. 자연어 개념명(`fee`)과 실제 CSV 컬럼명(`cu_charge_rt`)을 분리하면 원본 스키마가 바뀌어도 에이전트 계약을 유지하기 쉽습니다.

In [26]:
# ProductSpec은 도메인 용어와 실제 CSV 컬럼 사이의 허용된 매핑입니다.
@dataclass(frozen=True)
class ProductSpec:
    key: str
    label: str
    sample_file: str
    id_field: str
    name_field: str
    fields: dict[str, str]
    numeric_fields: frozenset[str]
    warnings: tuple[str, ...]

SPECS = {
    'domestic_bond': ProductSpec('domestic_bond', '국내채권', '국내채권_샘플.csv', 'PD_NO', 'PD_NM',
        {'name':'PD_NM', 'coupon_rate':'SRFC_IRT', 'buy_yield':'BUY_YIELD', 'after_tax_yield':'AFTER_TAX_YIELD'},
        frozenset({'coupon_rate','buy_yield','after_tax_yield'}), ('수익률은 값이 제공된 종목 안에서만 비교합니다.',)),
    'domestic_etf': ProductSpec('domestic_etf', '국내 ETF', '국내ETF_샘플.csv', 'pd_itm_no', 'pd_nm',
        {'name':'pd_nm', 'fee':'cu_charge_rt', 'aum':'du_last_aum', 'return_1m':'du_er_1m', 'return_1y':'du_er_1y', 'region':'wu_inv_rgn'},
        frozenset({'fee','aum','return_1m','return_1y'}), ('결측값을 0으로 해석하지 않습니다.',)),
    'overseas_etf': ProductSpec('overseas_etf', '해외 ETF', '해외ETF_샘플.csv', 'pd_itm_no', 'pd_nm',
        {'name':'pd_nm', 'ticker':'pd_abrv_nm', 'fee':'cu_charge_rt', 'aum':'du_last_aum', 'region':'wu_inv_rgn'},
        frozenset({'fee','aum'}), ('기초지수 미제공 문구는 실제 지수명으로 취급하지 않습니다.',)),
    'public_fund': ProductSpec('public_fund', '공모펀드', '공모펀드_샘플.csv', 'itm_no', 'itm_nm',
        {'name':'itm_nm', 'aum':'fd_nast_suma', 'return_1m':'fd_mm1_ern_r', 'return_1y':'fd_yr1_ern_r', 'region':'fd_ivst_rgn_desc'},
        frozenset({'aum','return_1m','return_1y'}), ('공모펀드 데이터에는 보수 정보가 없습니다.',)),
}
[(key, spec.sample_file, list(spec.fields)) for key, spec in SPECS.items()]

[('domestic_bond',
  '국내채권_샘플.csv',
  ['name', 'coupon_rate', 'buy_yield', 'after_tax_yield']),
 ('domestic_etf',
  '국내ETF_샘플.csv',
  ['name', 'fee', 'aum', 'return_1m', 'return_1y', 'region']),
 ('overseas_etf', '해외ETF_샘플.csv', ['name', 'ticker', 'fee', 'aum', 'region']),
 ('public_fund',
  '공모펀드_샘플.csv',
  ['name', 'aum', 'return_1m', 'return_1y', 'region'])]

## 4단계 — 금융상품 온톨로지 설정

온톨로지는 데이터 자체가 아니라 **개념, 동의어, 관계, 제약을 정의하는 설계도**입니다. 별도 Turtle 파일에는 상품 분류 계층을 기록하고, 실행에 필요한 최소 온톨로지는 Python 객체로 구성합니다.

- 동의어 정규화: `운용규모` → `AUM`, `수수료` → `총보수`
- 관계: ETF/채권/펀드는 금융상품의 하위 개념
- 제약: 상품군별로 실제 보유한 속성만 검색 가능
- 추적: Turtle 원본과 실행 규칙을 함께 확인 가능

In [12]:
@dataclass(frozen=True)
class FinancialProductOntology:
    # 다양한 사용자 표현을 질의 해석기가 아는 표준 용어로 통일합니다.
    aliases: dict[str, str]
    # is_a 관계는 상품 개념의 상하위 구조를 표현합니다.
    is_a: dict[str, str]
    # 상품군마다 데이터로 확인할 수 있는 속성만 명시합니다.
    available_fields: dict[str, frozenset[str]]

    def normalize_query(self, query: str) -> str:
        normalized = query
        # 긴 별칭부터 치환해 짧은 표현이 문장의 일부를 먼저 바꾸지 않게 합니다.
        for alias in sorted(self.aliases, key=len, reverse=True):
            normalized = re.sub(re.escape(alias), self.aliases[alias], normalized, flags=re.IGNORECASE)
        return normalized

    def validate_plan(self, plan: QueryPlan) -> None:
        # 온톨로지 제약을 실행 경계에서 검사해 존재하지 않는 속성 조회를 막습니다.
        if plan.product_type is None or plan.clarification:
            return
        allowed = self.available_fields[plan.product_type]
        requested = {item.field for item in plan.filters}
        if plan.sort_by:
            requested.add(plan.sort_by)
        unknown = requested - allowed
        if unknown:
            raise ValueError(f'{plan.product_type}에서 확인할 수 없는 온톨로지 속성: {sorted(unknown)}')

ONTOLOGY = FinancialProductOntology(
    aliases={'운용규모': 'AUM', '자산규모': 'AUM', '수수료': '총보수', '미 ETF': '해외 ETF'},
    is_a={
        'domestic_bond': 'bond', 'bond': 'financial_product',
        'domestic_etf': 'etf', 'overseas_etf': 'etf', 'etf': 'financial_product',
        'public_fund': 'fund', 'fund': 'financial_product',
    },
    available_fields={key: frozenset(spec.fields) for key, spec in SPECS.items()},
)

ttl_path = REPO_ROOT / 'agent_build_practice' / 'financial_product_ontology.ttl'
print('정규화:', ONTOLOGY.normalize_query('수수료가 낮은 미 ETF 3개'))
print('해외 ETF 상위 개념:', ONTOLOGY.is_a['overseas_etf'], '→', ONTOLOGY.is_a['etf'])
print('Turtle 온톨로지:', ttl_path)

정규화: 총보수가 낮은 해외 ETF 3개
해외 ETF 상위 개념: etf → financial_product
Turtle 온톨로지: /Users/park_jimin/Documents/ai패스티벌/agent_build_practice/financial_product_ontology.ttl


## 5단계 — 안전한 CSV 검색 도구

숫자 변환, 결측 처리, 필터, 정렬은 LLM이 아니라 코드가 수행합니다. 특히 결측값은 0이 아니며, 정렬 필드가 비어 있는 행은 후보에서 제외합니다. 각 결과에 원본 파일과 행 번호를 남겨 추적 가능하게 합니다.

In [27]:
EMPTY_VALUES = {'', 'null', 'none', 'nan'}

# 빈 문자열과 결측 표기를 None으로 통일하되 0은 유효한 값으로 보존합니다.
def clean(value: Any) -> str | None:
    if value is None:
        return None
    text = str(value).strip()
    return None if text.lower() in EMPTY_VALUES else text

def number(value: Any) -> float | None:
    text = clean(value)
    if text is None:
        return None
    try:
        return float(text.replace(',', ''))
    except ValueError:
        return None

class CsvProductAgent:
    def __init__(self, spec: ProductSpec):
        self.spec = spec
        self.csv_path = DATA_DIR / spec.sample_file

    # 필터 필드는 반드시 ProductSpec의 허용 목록을 통과해야 합니다.
    def _matches(self, row: dict[str, str], item: Filter) -> bool:
        if item.field not in self.spec.fields:
            raise ValueError(f'허용되지 않은 필드: {item.field}')
        raw = row.get(self.spec.fields[item.field])
        if item.operator == 'contains':
            return clean(raw) is not None and str(item.value).casefold() in str(raw).casefold()
        if item.operator == 'eq':
            return clean(raw) is not None and str(raw).strip().casefold() == str(item.value).strip().casefold()
        left, right = number(raw), number(item.value)
        if left is None or right is None:
            return False
        return left >= right if item.operator == 'gte' else left <= right

    def search(self, plan: QueryPlan) -> list[ProductResult]:
        if plan.sort_by and plan.sort_by not in self.spec.fields:
            raise ValueError(f'허용되지 않은 정렬 필드: {plan.sort_by}')
        with self.csv_path.open(encoding='utf-8-sig', newline='') as handle:
            candidates = [(row, n) for n, row in enumerate(csv.DictReader(handle), start=2)
                          if all(self._matches(row, f) for f in plan.filters)]
        # 정렬 기준이 결측인 행은 0으로 간주하지 않고 비교 대상에서 제외합니다.
        if plan.sort_by:
            column = self.spec.fields[plan.sort_by]
            numeric = plan.sort_by in self.spec.numeric_fields
            candidates = [(row, n) for row, n in candidates
                          if (number(row.get(column)) if numeric else clean(row.get(column))) is not None]
            candidates.sort(key=lambda x: number(x[0].get(column)) if numeric else clean(x[0].get(column)).casefold(),
                            reverse=plan.sort_order == 'desc')
        results = []
        for row, row_number in candidates[:plan.limit]:
            attrs = {name: clean(row.get(column)) for name, column in self.spec.fields.items()}
            results.append(ProductResult(self.spec.key, clean(row.get(self.spec.id_field)) or 'UNKNOWN',
                         clean(row.get(self.spec.name_field)) or '이름 미제공', attrs, self.spec.sample_file, row_number))
        return results

plan = interpreter.interpret('총보수가 낮은 해외 ETF 3개')
CsvProductAgent(SPECS['overseas_etf']).search(plan)

[ProductResult(product_type='overseas_etf', product_id='CNXT.K', name='VanEck ChiNext Innovators ETF', attributes={'name': 'VanEck ChiNext Innovators ETF', 'ticker': 'CNXT', 'fee': '0.000000', 'aum': '137630000.00', 'region': 'China'}, source_file='해외ETF_샘플.csv', source_row=2),
 ProductResult(product_type='overseas_etf', product_id='GIF', name='REX Growth & Income Universe ETF', attributes={'name': 'REX Growth & Income Universe ETF', 'ticker': 'GIF', 'fee': '0.000000', 'aum': '2790000.00', 'region': 'United States of America'}, source_file='해외ETF_샘플.csv', source_row=5),
 ProductResult(product_type='overseas_etf', product_id='LDRI.K', name='iShares iBonds 1-5 Year TIPS Ladder ETF', attributes={'name': 'iShares iBonds 1-5 Year TIPS Ladder ETF', 'ticker': 'LDRI', 'fee': '0.000000', 'aum': '20990000.00', 'region': 'United States of America'}, source_file='해외ETF_샘플.csv', source_row=38)]

## 6단계 — 전문 에이전트와 근거 검증 에이전트

상품군별 에이전트는 같은 검색 엔진을 재사용하되 서로 다른 스키마와 경고를 가집니다. 검증 에이전트는 검색 결과에서만 근거를 만들고, 상품 ID·원본 행·사용 필드를 함께 기록합니다.

In [28]:
class DomesticBondAgent(CsvProductAgent):
    def __init__(self): super().__init__(SPECS['domestic_bond'])
class DomesticEtfAgent(CsvProductAgent):
    def __init__(self): super().__init__(SPECS['domestic_etf'])
class OverseasEtfAgent(CsvProductAgent):
    def __init__(self): super().__init__(SPECS['overseas_etf'])
class PublicFundAgent(CsvProductAgent):
    def __init__(self): super().__init__(SPECS['public_fund'])

# 답변 근거를 검색 결과에서만 만들며 원본 파일과 행 번호를 보존합니다.
class EvidenceAgent:
    def build(self, spec: ProductSpec, results: list[ProductResult]):
        evidence = [{
            'product_id': item.product_id, 'product_name': item.name,
            'source_file': item.source_file, 'source_row': item.source_row,
            'used_fields': {k: v for k, v in item.attributes.items() if v is not None},
        } for item in results]
        warnings = list(spec.warnings) + ['조건 조회 결과이며 투자 권유가 아닙니다.']
        if not results:
            warnings.insert(0, '조건에 부합하면서 필요한 값이 있는 상품을 찾지 못했습니다.')
        return evidence, warnings

## 7단계 — 오케스트레이터로 연결

오케스트레이터는 상태를 읽고 다음 노드를 고릅니다. 이 예제의 경로는 `interpret → clarify 또는 route → search → verify → respond`입니다. 고정된 업무 절차이므로 자유로운 에이전트 루프보다 명시적인 워크플로가 안전합니다.

In [29]:
class FinancialProductOrchestrator:
    AGENTS = {
        'domestic_bond': DomesticBondAgent, 'domestic_etf': DomesticEtfAgent,
        'overseas_etf': OverseasEtfAgent, 'public_fund': PublicFundAgent,
    }
    def __init__(self):
        self.interpreter = QueryInterpreterAgent()
        self.ontology = ONTOLOGY
        self.evidence_agent = EvidenceAgent()

    def answer(self, query: str) -> AgentResponse:
        # 1) 온톨로지 동의어로 정규화한 뒤 2) 실행 계획을 만들고 3) 제약을 검증합니다.
        normalized_query = self.ontology.normalize_query(query)
        plan = self.interpreter.interpret(normalized_query)
        plan.original_query = query  # 사용자 원문은 근거 추적을 위해 다시 보존합니다.
        self.ontology.validate_plan(plan)
        if plan.clarification:
            return AgentResponse(plan.clarification, plan, warnings=['추가 조건이 필요합니다.'])
        spec = SPECS[plan.product_type]
        # 검색과 수치 정렬은 LLM이 아니라 결정론적인 전문 에이전트가 수행합니다.
        results = self.AGENTS[plan.product_type]().search(plan)
        evidence, warnings = self.evidence_agent.build(spec, results)
        if not results:
            answer = f'조건에 부합하는 {spec.label} 상품을 찾지 못했습니다.'
        else:
            lines = [f'{spec.label} 조회 결과 {len(results)}건입니다.']
            for i, item in enumerate(results, 1):
                value = item.attributes.get(plan.sort_by) if plan.sort_by else item.product_id
                lines.append(f"{i}. {item.name} — {plan.sort_by or '상품 ID'}: {value}")
            answer = '\n'.join(lines)
        return AgentResponse(answer, plan, results, evidence, warnings)

app = FinancialProductOrchestrator()
response = app.answer('총보수가 낮은 해외 ETF 3개')
print(response.answer)
print('\n근거 예시:')
print(json.dumps(response.evidence[0], ensure_ascii=False, indent=2))
print('\n경고:', *response.warnings, sep='\n- ')

해외 ETF 조회 결과 3건입니다.
1. VanEck ChiNext Innovators ETF — fee: 0.000000
2. REX Growth & Income Universe ETF — fee: 0.000000
3. iShares iBonds 1-5 Year TIPS Ladder ETF — fee: 0.000000

근거 예시:
{
  "product_id": "CNXT.K",
  "product_name": "VanEck ChiNext Innovators ETF",
  "source_file": "해외ETF_샘플.csv",
  "source_row": 2,
  "used_fields": {
    "name": "VanEck ChiNext Innovators ETF",
    "ticker": "CNXT",
    "fee": "0.000000",
    "aum": "137630000.00",
    "region": "China"
  }
}

경고:
- 기초지수 미제공 문구는 실제 지수명으로 취급하지 않습니다.
- 조건 조회 결과이며 투자 권유가 아닙니다.


## 8단계 — 경계 사례로 가드레일 확인

정상 질문만 시험하면 안전성을 알 수 없습니다. 상품군 누락, 데이터에 없는 필드, 과도한 결과 수, 결측값 정렬을 확인합니다. 아래 셀은 모두 통과해야 합니다.

In [30]:
assert app.answer('수익률 높은 상품 5개').query_plan.clarification
assert '보수 정보가 없습니다' in app.answer('보수가 낮은 공모펀드 3개').answer
assert app.answer('AUM 높은 국내 ETF 100개').query_plan.limit == 20
assert all(item.attributes['fee'] is not None for item in app.answer('총보수가 낮은 해외 ETF 5개').results)
assert app.answer('수수료가 낮은 미 ETF 3개').query_plan.product_type == 'overseas_etf'
try:
    OverseasEtfAgent().search(QueryPlan('overseas_etf', sort_by='secret_column'))
    raise AssertionError('허용 목록 검사가 작동하지 않았습니다.')
except ValueError as exc:
    print('허용 목록 차단 성공:', exc)
print('모든 경계 사례를 통과했습니다.')

허용 목록 차단 성공: 허용되지 않은 정렬 필드: secret_column
모든 경계 사례를 통과했습니다.


## 9단계 — 기존 `financial_product_agents`와 비교

지금 만든 축약판과 저장소의 모듈형 구현을 같은 질문으로 실행합니다. 운영 코드는 더 넓은 필드 카탈로그와 별도 모듈·테스트를 갖고 있습니다.

In [31]:
PROJECT_DIR = REPO_ROOT / 'financial_product_agents'
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from agents.orchestrator import FinancialProductOrchestrator as ExistingOrchestrator

query = '매수수익률이 높은 국내채권 2개'
existing = ExistingOrchestrator(REPO_ROOT).answer(query)
print(existing.answer)
print('\n근거 수:', len(existing.evidence))
print('데이터 기준일:', existing.data_as_of)

국내채권 `buy_yield` 기준 조회 결과 1건입니다.
1. 부산지역개발채권 22-09 — buy_yield: 3.04

근거 수: 1
데이터 기준일: 2026-07-11


## 10단계 — OpenAI LLM 연결부

LLM 설정과 연결을 다른 에이전트 코드에서 분리합니다. OpenAI 공식 Python SDK와 Responses API를 사용하며, API 키는 코드에 적지 않고 `OPENAI_API_KEY` 환경변수 또는 마스킹 입력으로 받습니다.

가상환경에서 처음 한 번만 아래 설치 셀을 실행하세요. `%pip`는 현재 노트북 커널에 패키지를 설치합니다.

In [ ]:
%pip install -q --upgrade openai

### LLM 설정 셀

모델명과 API 키 입력 방식은 이 셀에서만 관리합니다. 실제 연결을 원하면 `ENABLE_OPENAI = True`로 변경하세요. 키는 출력하거나 노트북에 저장하지 않습니다.

In [ ]:
import getpass
import os
from openai import OpenAI

OPENAI_MODEL = os.getenv('OPENAI_MODEL', 'gpt-4o-mini')
ENABLE_OPENAI = False  # 실제 API 호출과 과금에 동의할 때 True로 변경

def load_openai_api_key() -> str:
    key = os.getenv('OPENAI_API_KEY', '').strip()
    if not key:
        key = getpass.getpass('OpenAI API Key (화면에 표시되지 않음): ').strip()
    if not key:
        raise RuntimeError('OPENAI_API_KEY가 필요합니다.')
    return key

print('OpenAI 모델:', OPENAI_MODEL)
print('실제 연결 활성화:', ENABLE_OPENAI)

In [ ]:
class OpenAILLMClient:
    def __init__(self, api_key: str, model: str = OPENAI_MODEL):
        self.client = OpenAI(api_key=api_key)
        self.model = model

    def chat(self, system: str, user: str, *, json_schema: dict | None = None, max_tokens: int = 800) -> str:
        request = {
            'model': self.model, 'instructions': system, 'input': user,
            'max_output_tokens': max_tokens,
        }
        if json_schema is not None:
            # Responses API의 Structured Outputs로 QueryPlan JSON 형식을 고정합니다.
            request['text'] = {'format': {
                'type': 'json_schema', 'name': 'query_plan',
                'schema': json_schema, 'strict': True,
            }}
        response = self.client.responses.create(**request)
        return response.output_text

QUERY_PLAN_SCHEMA = {
    'type': 'object',
    'properties': {
        'product_type': {'type': ['string','null'], 'enum': ['domestic_bond','domestic_etf','overseas_etf','public_fund',None]},
        'filters': {'type': 'array', 'items': {'type': 'object', 'properties': {
            'field': {'type':'string'}, 'operator': {'type':'string','enum':['eq','contains','gte','lte']},
            'value': {'type':['string','number']}}, 'required':['field','operator','value'], 'additionalProperties':False}},
        'sort_by': {'type':['string','null']}, 'sort_order': {'type':'string','enum':['asc','desc']},
        'limit': {'type':'integer','minimum':1,'maximum':20}, 'clarification': {'type':['string','null']},
    },
    'required':['product_type','filters','sort_by','sort_order','limit','clarification'],
    'additionalProperties': False,
}

### LLM 연결 여부 확인 — 질의 해석기 사용 직전

다음 셀은 실제 모델 객체를 만들고 최소 요청으로 인증·모델 접근 여부를 확인합니다. 이후 LLM 필요 구간에서는 `LLM_CONNECTED`를 먼저 검사합니다.

In [ ]:
llm_client = None
LLM_CONNECTED = False

if ENABLE_OPENAI:
    try:
        llm_client = OpenAILLMClient(load_openai_api_key())
        check = llm_client.chat('한 단어로만 답하세요.', 'OK라고 답하세요.', max_tokens=10)
        LLM_CONNECTED = bool(check.strip())
        print('LLM 연결 성공:', check.strip())
    except Exception as exc:
        print('LLM 연결 실패:', type(exc).__name__, str(exc)[:300])
else:
    print('LLM 미연결 — ENABLE_OPENAI=True로 바꾸고 이 셀을 다시 실행하세요.')

print('LLM_CONNECTED =', LLM_CONNECTED)

### LLM 사용 1 — 자연어 질의를 구조화된 계획으로 변환

AI가 만든 JSON은 그대로 실행하지 않고 기존 `QueryPlan`과 온톨로지 검증을 통과시킵니다.

In [ ]:
class AIQueryInterpreterAgent:
    def __init__(self, client, ontology):
        self.client, self.ontology = client, ontology
    def interpret(self, query: str) -> QueryPlan:
        normalized = self.ontology.normalize_query(query)
        catalog = {key: sorted(spec.fields) for key, spec in SPECS.items()}
        system = ('금융상품 질의를 검색 계획 JSON으로 변환한다. 제공된 상품군과 필드만 사용한다. '
                  '낮은/저렴한은 asc, 높은/많은은 desc다. 부족하면 clarification에 질문을 쓴다.')
        raw = self.client.chat(system, f'허용 카탈로그: {json.dumps(catalog, ensure_ascii=False)}\n질의: {normalized}',
                               json_schema=QUERY_PLAN_SCHEMA)
        payload = json.loads(raw)
        plan = QueryPlan(payload.get('product_type'),
            [Filter(x['field'], x['operator'], x['value']) for x in payload.get('filters', [])],
            payload.get('sort_by'), payload.get('sort_order','desc'), payload.get('limit',5),
            payload.get('clarification'), query)
        self.ontology.validate_plan(plan)
        return plan

### LLM 연결 여부 확인 — 근거 기반 답변 생성기 사용 직전

In [ ]:
if LLM_CONNECTED:
    print('LLM 연결됨 — 근거 기반 답변 생성기를 사용할 수 있습니다.')
else:
    print('LLM 미연결 — 아래 라이브 실행은 건너뛰고 결정론적/모의 테스트만 사용합니다.')

In [ ]:
class AIEvidenceAnswerAgent:
    def __init__(self, client): self.client = client
    def generate(self, query, results, evidence, warnings) -> str:
        context = {'query':query, 'data_as_of':'2026-07-11', 'results':[asdict(x) for x in results],
                   'evidence':evidence, 'warnings':warnings}
        system = ('전달된 context의 사실과 수치만 사용해 한국어로 설명한다. 추천이나 미래 예측을 하지 않는다. '
                  '각 상품에 source_file과 source_row를 표시하고, 없는 내용은 확인할 수 없다고 말한다.')
        return self.client.chat(system, json.dumps(context, ensure_ascii=False), max_tokens=1200)

class AIFinancialProductOrchestrator:
    def __init__(self, client):
        self.interpreter = AIQueryInterpreterAgent(client, ONTOLOGY)
        self.evidence_agent = EvidenceAgent()
        self.answer_agent = AIEvidenceAnswerAgent(client)
    def answer(self, query: str) -> AgentResponse:
        plan = self.interpreter.interpret(query)
        if plan.clarification:
            return AgentResponse(plan.clarification, plan, warnings=['추가 조건이 필요합니다.'])
        agents = {'domestic_bond':DomesticBondAgent, 'domestic_etf':DomesticEtfAgent,
                  'overseas_etf':OverseasEtfAgent, 'public_fund':PublicFundAgent}
        spec = SPECS[plan.product_type]
        results = agents[plan.product_type]().search(plan)
        evidence, warnings = self.evidence_agent.build(spec, results)
        return AgentResponse(self.answer_agent.generate(query, results, evidence, warnings),
                             plan, results, evidence, warnings)

### LLM 연결 여부 확인 — 전체 에이전트 실행 직전

In [ ]:
print('전체 에이전트 실행 가능 여부:', '가능' if LLM_CONNECTED else '불가 — LLM 미연결')

In [ ]:
if LLM_CONNECTED:
    live_response = AIFinancialProductOrchestrator(llm_client).answer(
        '총보수가 낮은 해외 ETF 3개를 근거와 함께 알려줘'
    )
    print('AI 검색 계획:', json.dumps(asdict(live_response.query_plan), ensure_ascii=False, indent=2))
    print('검색 결과/근거:', len(live_response.results), '/', len(live_response.evidence))
    print('최종 답변:\n', live_response.answer)
else:
    print('LLM 연결 후 이 셀을 다시 실행하세요.')

## 직접 확장해보기

다음 순서로 확장하면 좋습니다.

1. `QueryInterpreterAgent`에 `gte`/`lte` 수치 조건(예: ‘보수 0.5 이하’)을 추가합니다.
2. 질의 해석 결과를 JSON 평가 세트로 저장하고 예상 `QueryPlan`과 비교합니다.
3. 검색 결과에 없는 숫자가 최종 답변에 포함되지 않는지 자동 평가합니다.
4. API 재시도·타임아웃·사용량 로깅과 비밀정보 마스킹을 운영 수준으로 확장합니다.
5. 온톨로지/지식 그래프를 추가할 때도 수치 정렬은 CSV 또는 DB가 담당하게 유지합니다.

핵심은 모델을 먼저 붙이는 것이 아니라 **계약, 도구, 근거, 실패 경로를 먼저 고정하는 것**입니다.